# Future Stars Baseball Club - Hitting 2D Pose Measurement Prototype

This notebook invokes the original V3 MediaPipe/OpenCV script to overlay an image-plane skeleton and export exploratory 2D angles. It does not perform validated 3D biomechanics, automatic scoring, multi-view fusion, AI reporting, or retest evaluation.

Keep the repository files together, with src/process_hitting_video.py and requirements.txt present. Opening this notebook alone in Colab does not copy those files: add them to those paths first.

The legacy mp.solutions.pose API is required. Use the pinned requirements with Python 3.12; see docs/limitations.md. The script was smoke-tested locally, but this notebook has not been run end to end in Colab.

Colab uploads share files with Google. For a public demo, use only non-identifying test media you have permission to upload. Do not use participant videos. Keep notebook outputs private and clear them before committing.


In [ ]:
# 1. Locate the repository; install the pinned legacy runtime.
from pathlib import Path
import os, sys
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12 for the tested legacy runtime.'
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src/process_hitting_video.py').is_file() and (p / 'requirements.txt').is_file()), None)
assert repo_root is not None, 'Add src/process_hitting_video.py and requirements.txt to the runtime before continuing.'
os.chdir(repo_root)
%pip install -r requirements.txt
import mediapipe as mp
assert hasattr(mp, 'solutions') and hasattr(mp.solutions, 'pose'), 'Legacy API unavailable; restart the runtime after installing the pinned requirements.'


In [ ]:
# 2. Upload non-identifying test video(s) to the Colab runtime
from google.colab import files
import os

os.makedirs('input_videos', exist_ok=True)
os.makedirs('analyzed_output', exist_ok=True)

print('Upload your hitting video(s) below (e.g. synthetic_demo_side_tee.mp4):')
uploaded = files.upload()
for fn in uploaded.keys():
    os.rename(fn, os.path.join('input_videos', fn))
print('Uploaded files:', os.listdir('input_videos'))

In [ ]:
# 3. Process into a fresh directory; stop on a failing exit status.
import subprocess, tempfile
run_succeeded = False
run_output = Path(tempfile.mkdtemp(prefix='run_', dir='analyzed_output'))
subprocess.run([sys.executable, 'src/process_hitting_video.py', '--input_dir', './input_videos', '--output_dir', str(run_output)], check=True)
run_succeeded = True


In [ ]:
# 4. Download only this successful run; keep outputs private.
assert globals().get('run_succeeded', False), 'The pipeline has not completed successfully.'
for f in sorted(run_output.iterdir()):
    if f.is_file() and f.suffix in {'.mp4', '.csv'}:
        files.download(str(f))
